In [1]:
!pip install -q groq PyPDF2 chromadb gradio

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 60.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 22.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 87.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 69.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.

In [7]:
from PyPDF2 import PdfReader
import chromadb
from groq import Groq
from google.colab import files
import gradio as gr

# ---- YOUR GROQ KEY ----
GROQ_API_KEY = "your-groq-api-key-here"  # replace!
groq_client = Groq(api_key=GROQ_API_KEY)

# ---- UPLOAD PDF ----
print("📄 Upload your research paper PDF...")
uploaded = files.upload()
filename = list(uploaded.keys())[0]

# ---- READ PDF ----
reader = PdfReader(filename)
full_text = ""
for i, page in enumerate(reader.pages):
    text = page.extract_text()
    if text:
        full_text += f"\n[Page {i+1}]\n{text}"
print(f"✅ PDF loaded! Pages: {len(reader.pages)}")

# ---- CHUNK TEXT ----
def chunk_text(text, chunk_size=800, overlap=100):
    words = text.split()
    chunks = []
    i = 0
    while i < len(words):
        chunk = " ".join(words[i:i+chunk_size])
        chunks.append(chunk)
        i += chunk_size - overlap
    return chunks

chunks = chunk_text(full_text)
print(f"✅ Split into {len(chunks)} chunks")

# ---- STORE IN VECTOR DB ----
chroma_client = chromadb.Client()
try:
    chroma_client.delete_collection("research_paper")
except:
    pass
collection = chroma_client.create_collection("research_paper")
for i, chunk in enumerate(chunks):
    collection.add(documents=[chunk], ids=[f"chunk_{i}"])
print(f"✅ Stored in ChromaDB!")

# ---- ASK QUESTIONS ----
def chatbot(question):
    if not question.strip():
        return "Please ask a question!"
    results = collection.query(query_texts=[question], n_results=3)
    source_chunks = results['documents'][0]
    context = "\n\n".join(source_chunks)
    prompt = f"""You are an AI research assistant for an Emergency Detection System paper.
Answer using ONLY the context below. Always cite which page supports your answer.
Context:
{context}
Question: {question}
Answer:"""
    response = groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=400,
        temperature=0.1
    )
    answer = response.choices[0].message.content
    evidence = "\n\n".join([f"📄 ...{c[:250]}..." for c in source_chunks])
    return f"## 💡 Answer\n{answer}\n\n---\n## 🔍 Evidence\n{evidence}"

def quick_q1(): return chatbot("What problem does this research solve?")
def quick_q2(): return chatbot("What accuracy did the models achieve?")
def quick_q3(): return chatbot("What is decision level fusion?")
def quick_q4(): return chatbot("What APIs were used for sending alerts?")
def quick_q5(): return chatbot("What datasets were used for training?")

custom_css = """
body { background: #0f0f1a !important; }
.gradio-container {
    background: linear-gradient(135deg, #0f0f1a 0%, #1a1a2e 100%) !important;
    font-family: 'Segoe UI', sans-serif !important;
}
#header {
    background: linear-gradient(90deg, #e63946, #c1121f);
    border-radius: 16px;
    padding: 24px;
    text-align: center;
    margin-bottom: 20px;
    box-shadow: 0 4px 20px rgba(230,57,70,0.4);
}
#header h1 { color: white !important; font-size: 2em; margin: 0; }
#header p { color: #ffd6d6 !important; margin: 6px 0 0 0; font-size: 1em; }
footer { display: none !important; }
"""
with gr.Blocks(css="""
body { background: #0a0f1e !important; }
.gradio-container {
    background: #0a0f1e !important;
    font-family: 'Segoe UI', sans-serif !important;
    max-width: 900px !important;
    margin: auto !important;
}
footer { display: none !important; }
.message-box {
    background: #0d1b2a;
    border: 1px solid #1e3a5f;
    border-radius: 16px;
    padding: 20px;
    margin: 8px 0;
    color: #e3f2fd;
}
.user-message {
    background: #1a3a6b;
    border: none;
    border-radius: 16px;
    padding: 14px 20px;
    margin: 8px 0 8px 80px;
    color: white;
}
.bot-message {
    background: #0d1b2a;
    border: 1px solid #1e3a5f;
    border-radius: 16px;
    padding: 14px 20px;
    margin: 8px 80px 8px 0;
    color: #e3f2fd;
}
textarea {
    background: #0d1b2a !important;
    border: 1px solid #1e56b0 !important;
    color: white !important;
    border-radius: 12px !important;
    font-size: 1em !important;
}
button.primary {
    background: linear-gradient(90deg, #1a3a6b, #0d47a1) !important;
    border: none !important;
    color: white !important;
    border-radius: 10px !important;
    font-weight: bold !important;
}
.quick-btn {
    background: #0d1b2a !important;
    border: 1px solid #1e56b0 !important;
    color: #90caf9 !important;
    border-radius: 20px !important;
    font-size: 0.82em !important;
    padding: 4px 12px !important;
}
.quick-btn:hover {
    background: #1a3a6b !important;
    color: white !important;
}
label { color: #90caf9 !important; }
""", title="Emergency Q&A") as demo:

    # ---- HEADER ----
    gr.HTML("""
    <div style="text-align:center; padding:24px 0 8px 0;">
        <div style="font-size:2.5em;">🚨</div>
        <h2 style="color:white; margin:8px 0 4px 0;">Emergency Detection Q&A</h2>
        <p style="color:#546e7a; font-size:0.85em; margin:0;">
        RAG · ChromaDB · Llama 3.3 · Built by Bokka Sowjanya · IDCIoT 2026
        </p>
    </div>
    """)

    # ---- CHAT WINDOW ----
    chatbot_ui = gr.Chatbot(
        value=[
            [None, "👋 Hi Sowjanya! I'm your Research Assistant.\n\nI've read your paper **'Smart AI Multimodal Architecture for Automated Emergency Detection and Alerts'** published at IDCIoT 2026.\n\nAsk me anything about it! 🎓"]
        ],
        height=450,
        bubble_full_width=False,
        show_label=False,
        container=True,
        avatar_images=(None, "https://api.dicebear.com/7.x/bottts/svg?seed=emergency")
    )

    # ---- QUICK BUTTONS ----
    gr.HTML("<p style='color:#546e7a; font-size:0.82em; margin:8px 4px 4px 4px;'>⚡ Quick Questions:</p>")
    with gr.Row():
        btn1 = gr.Button("What problem?", elem_classes="quick-btn")
        btn2 = gr.Button("Accuracy?", elem_classes="quick-btn")
        btn3 = gr.Button("Decision fusion?", elem_classes="quick-btn")
        btn4 = gr.Button("APIs used?", elem_classes="quick-btn")
        btn5 = gr.Button("Technologies?", elem_classes="quick-btn")

    # ---- INPUT BOX ----
    gr.HTML('<div style="height:8px"></div>')
    with gr.Row():
        with gr.Column(scale=5):
            msg_input = gr.Textbox(
                placeholder="Message Emergency Research Assistant...",
                lines=1,
                label="",
                container=False,
                show_label=False
            )
        with gr.Column(scale=1, min_width=100):
            send_btn = gr.Button("Send ➤", variant="primary")

    # ---- FOOTER ----
    gr.HTML("""
    <div style="text-align:center; padding:12px; color:#37474f; font-size:0.75em;">
    Answers based only on your research paper · No hallucination · Every answer is cited
    </div>
    """)

    # ---- CHAT FUNCTION ----
    def respond(message, chat_history):
        if not message.strip():
            return "", chat_history

        results = collection.query(query_texts=[message], n_results=3)
        source_chunks = results['documents'][0]
        context = "\n\n".join(source_chunks)

        prompt = f"""You are an AI research assistant for the paper:
"Smart AI Multimodal Architecture for Automated Emergency Detection and Alerts"
published at IDCIoT 2026 by Bokka Sowjanya et al.

Answer using ONLY the context below. Always cite the page number.
Be concise and clear.

Context:
{context}

Question: {message}
Answer:"""

        response = groq_client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[{"role": "user", "content": prompt}],
            max_tokens=400,
            temperature=0.1
        )
        answer = response.choices[0].message.content
        chat_history.append((message, answer))
        return "", chat_history

    def quick(q, history):
        return respond(q, history)

    send_btn.click(fn=respond, inputs=[msg_input, chatbot_ui], outputs=[msg_input, chatbot_ui])
    msg_input.submit(fn=respond, inputs=[msg_input, chatbot_ui], outputs=[msg_input, chatbot_ui])
    btn1.click(fn=lambda h: quick("What problem does this research solve?", h), inputs=chatbot_ui, outputs=[msg_input, chatbot_ui])
    btn2.click(fn=lambda h: quick("What accuracy did the models achieve?", h), inputs=chatbot_ui, outputs=[msg_input, chatbot_ui])
    btn3.click(fn=lambda h: quick("What is decision level fusion?", h), inputs=chatbot_ui, outputs=[msg_input, chatbot_ui])
    btn4.click(fn=lambda h: quick("What APIs were used for sending alerts?", h), inputs=chatbot_ui, outputs=[msg_input, chatbot_ui])
    btn5.click(fn=lambda h: quick("What technologies and tools were used?", h), inputs=chatbot_ui, outputs=[msg_input, chatbot_ui])

demo.launch(share=True)

📄 Upload your research paper PDF...


Saving Smart_AI_Multimodal_Architecture_for_Automated_Emergency_Detection_and_Alerts.pdf to Smart_AI_Multimodal_Architecture_for_Automated_Emergency_Detection_and_Alerts (4).pdf
✅ PDF loaded! Pages: 7
✅ Split into 7 chunks
✅ Stored in ChromaDB!


/tmp/ipykernel_2690/3116102955.py:97: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css="""
/tmp/ipykernel_2690/3116102955.py:171: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot_ui = gr.Chatbot(
/tmp/ipykernel_2690/3116102955.py:171: DeprecationWarning: The 'bubble_full_width' parameter will be removed in Gradio 6.0. This parameter no longer has any effect.
  chatbot_ui = gr.Chatbot(
/tmp/ipykernel_2690/3116102955.py:171: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to 

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://19cf922b53bc19cc11.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
